In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from view import *
from show import *
from utils import *
from transforms import *
from processing import process

In [14]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

flat_file = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))[-4]

print(flat_file)

/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits


In [15]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/pmi/20250919/fdt/*'))
files_fdt

['/home/ulyanov/data/pmi/20250919/fdt/phi-fdt-blos_20250919T051503_V202611021000C_0549190501.fits',
 '/home/ulyanov/data/pmi/20250919/fdt/phi-fdt-blos_20250919T111503_V202611021003C_0549190502.fits',
 '/home/ulyanov/data/pmi/20250919/fdt/phi-fdt-blos_20250919T171503_V202611021006C_0549190503.fits',
 '/home/ulyanov/data/pmi/20250919/fdt/phi-fdt-blos_20250919T220003_V202611021009C_0549190515.fits',
 '/home/ulyanov/data/pmi/20250919/fdt/solo_L1_phi-fdt-ilam_20250919T213003_V202604242053C_0549190514.fits.gz']

In [16]:
#with fits.open(files_fdt[-1]) as hdul:
#    header_fdt = hdul[0].header
#    data_fdt = hdul[0].data

data_fdt, header_fdt = process(files_fdt[-1],
                               dark_file=dark_file,
                               flatfield_file=flat_file,
                               distortion_file=distortion_file,
                               _find_center=True)

#header_fdt['CROTA'] += 0.25



In [17]:
header_fdt['CROTA'] += 0.25

In [18]:
nx, ny = 1024, 1024
xc, yc = (nx - 1) / 2, (ny - 1) / 2
rsun = 0.45 * nx

data_fdt_, view_fdt = reproject(data_fdt[0,0], header_fdt, kind='',
                                nx=2048, ny=2048,
                                xc=1024, yc=1024, rsun = 1000,
                               crota=0, rsun_arc=0, #crlt=30,
                               minmu=0.)

In [28]:
show_data(data_fdt_, view_fdt,
          title='FDT', label=r'',
          cmap='inferno', vmin=0, vmax=2e4
          )

(<Figure size 1706x1706 with 2 Axes>, <Axes: title={'center': 'FDT'}>)

In [20]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/pmi/20250919/hmi/*'))

files_hmi

['/home/ulyanov/data/pmi/20250919/hmi/hmi.Ic_45s.20250919_213600_TAI.2.continuum.fits',
 '/home/ulyanov/data/pmi/20250919/hmi/hmi.M_720s.20250919_052400_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/pmi/20250919/hmi/hmi.M_720s.20250919_112400_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/pmi/20250919/hmi/hmi.M_720s.20250919_172400_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/pmi/20250919/hmi/hmi.M_720s.20250919_220000_TAI.3.magnetogram.fits']

In [79]:
with fits.open(files_hmi[0]) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data


#data_hmi = rebin(data_hmi, 2, update_header=header_hmi)

In [84]:
view_hmi = View.from_header(header_hmi).update(minmu=0.1, powmu=0).update(multiply=True, rsun=1)
data_hmi_ = view_fdt.update(powmu=0).reproject(data_hmi, view_hmi)

In [81]:
show_data(data_hmi_, view_fdt,
          title='HMI (reprojected)', label=r'',
          cmap='inferno', vmin=0, vmax=6e4
          )

(<Figure size 1706x1706 with 2 Axes>,
 <Axes: title={'center': 'HMI (reprojected)'}>)

In [62]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt_[1100:1300,1200:1400], cmap='inferno')
plt.tight_layout()

In [85]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi_[1100:1300,1200:1400], cmap='inferno')
plt.tight_layout()

In [82]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt_[1100:1300,1200:1400] - data_hmi_[1100:1300,1200:1400] / 2.4, cmap='inferno', vmin=-2000, vmax=2000)
plt.tight_layout()